# Machine Learning: LightGBM & CatBoost (n1)

**Objective**: Train LightGBM and CatBoost classifiers combining:
1. **TF-IDF embeddings** (900-dim from strict text) - GENERATED
2. **PhoBERT v2 Pretrained** (768-dim Layer 12 from loose text) - EXTRACTED
3. **ALL additional hand-crafted features** - EXTRACTED FROM DATASET

**Strategy**:
- Generate TF-IDF embeddings on-the-fly
- Extract PhoBERT embeddings from pre-trained model directly
- Extract ALL non-text columns as hand-crafted features
- Combine via concatenation
- Train LightGBM and CatBoost with StandardScaler + balanced weights
- Evaluate on validation set
- Save model + scaler + feature importance

## 1. Import Libraries & Device Setup

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import joblib
import torch
import gc
from tqdm import tqdm
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve
)
from transformers import AutoTokenizer, AutoModel

# Setup
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"✅ Libraries imported successfully")
print(f"   Device: {device}")

✅ Libraries imported successfully
   Device: cuda


## 2. Load Data

In [3]:
print("="*80)
print("LOADING DATA")
print("="*80)

# Load train/val sets
train_path = '../../../data/splited/train_set.csv'
val_path = '../../../data/splited/val_set.csv'
test_path = '../../../data/splited/test_set.csv'

df_train = pd.read_csv(train_path)
df_val = pd.read_csv(val_path)
df_test = pd.read_csv(test_path)

print(f"\nData loaded:")
print(f"  Train: {df_train.shape[0]} samples × {df_train.shape[1]} columns")
print(f"  Val:   {df_val.shape[0]} samples × {df_val.shape[1]} columns")
print(f"  Test:  {df_test.shape[0]} samples × {df_test.shape[1]} columns")
print(f"\nAvailable columns ({len(df_train.columns)}):")
for i, col in enumerate(df_train.columns, 1):
    print(f"  {i:2d}. {col}")

# Extract labels
y_train = df_train['label'].values
y_val = df_val['label'].values
y_test = df_test['label'].values


print(f"\nLabels:")
print(f"  Train: {(y_train == 0).sum()} fake / {(y_train == 1).sum()} real")
print(f"  Val:   {(y_val == 0).sum()} fake / {(y_val == 1).sum()} real")
print(f"  Test:  {(y_test == 0).sum()} fake / {(y_test == 1).sum()} real")

LOADING DATA

Data loaded:
  Train: 3788 samples × 28 columns
  Val:   474 samples × 28 columns
  Test:  474 samples × 28 columns

Available columns (28):
   1. id
   2. post_message
   3. label
   4. num_char
   5. num_emoji
   6. num_url
   7. num_hashtag
   8. num_like
   9. num_cmt
  10. num_share
  11. text_strict
  12. text_loose
  13. feat_word_count
  14. feat_exclamation
  15. feat_question
  16. feat_uppercase_words
  17. feat_uppercase_ratio
  18. feat_punctuation_ratio
  19. feat_engagement_total
  20. feat_like_ratio
  21. feat_comment_ratio
  22. feat_like_per_char
  23. feat_hour_sin
  24. feat_hour_cos
  25. feat_day_sin
  26. feat_day_cos
  27. feat_month_sin
  28. feat_month_cos

Labels:
  Train: 3143 fake / 645 real
  Val:   393 fake / 81 real
  Test:  393 fake / 81 real


## 3. Generate TF-IDF Embeddings (900-dim)

In [4]:
print("\n" + "="*80)
print("GENERATING TF-IDF EMBEDDINGS (900-dim)")
print("="*80)

# Extract strict text for TF-IDF
texts_train_strict = df_train['text_strict'].fillna('').tolist()
texts_val_strict = df_val['text_strict'].fillna('').tolist()
texts_test_strict = df_test['text_strict'].fillna('').tolist()

print(f"\nGenerating TF-IDF from strict text...")
print(f"  Train texts: {len(texts_train_strict)}")
print(f"  Val texts: {len(texts_val_strict)}")
print(f"  Test texts: {len(texts_test_strict)}")


# Create FULL TF-IDF vectorizer (same config as n0_TFIDF_statistical_methods)
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_df=0.95,
    min_df=2,
    sublinear_tf=True
)

# Fit on train, transform both
X_train_tfidf_full = tfidf.fit_transform(texts_train_strict)
X_val_tfidf_full = tfidf.transform(texts_val_strict)
X_test_tfidf_full = tfidf.transform(texts_test_strict)
print(f"\nFull TF-IDF vocabulary size: {X_train_tfidf_full.shape[1]}")

# Apply TruncatedSVD to reduce to 900 dims
svd = TruncatedSVD(n_components=900, random_state=42)
X_train_tfidf = svd.fit_transform(X_train_tfidf_full)
X_val_tfidf = svd.transform(X_val_tfidf_full)
X_test_tfidf = svd.transform(X_test_tfidf_full)

variance_retained = svd.explained_variance_ratio_.sum()

print(f"\n✅ TF-IDF embeddings generated:")
print(f"   Train: {X_train_tfidf.shape}")
print(f"   Val: {X_val_tfidf.shape}")
print(f"   Test: {X_test_tfidf.shape}")

print(f"   Variance retained: {variance_retained:.2%}")


GENERATING TF-IDF EMBEDDINGS (900-dim)

Generating TF-IDF from strict text...
  Train texts: 3788
  Val texts: 474
  Test texts: 474

Full TF-IDF vocabulary size: 57103

✅ TF-IDF embeddings generated:
   Train: (3788, 900)
   Val: (474, 900)
   Test: (474, 900)
   Variance retained: 49.05%


## 4. Extract PhoBERT Embeddings (Layer 12)

In [5]:
print("\n" + "="*80)
print("EXTRACTING PHOBERT EMBEDDINGS (Layer 12 - Pretrained)")
print("="*80)

# Extract loose text for PhoBERT
texts_train_loose = df_train['text_loose'].fillna('').tolist()
texts_val_loose = df_val['text_loose'].fillna('').tolist()
texts_test_loose = df_test['text_loose'].fillna('').tolist()

def extract_phobert_embeddings(texts, batch_size=16):
    """
    Extract [CLS] token embeddings from Layer 12 of PhoBERT v2 Pretrained model.
    Returns: numpy array of shape (num_texts, 768)
    """
    print(f"    Loading model...")
    tokenizer = AutoTokenizer.from_pretrained('vinai/phobert-base-v2')
    model = AutoModel.from_pretrained('vinai/phobert-base-v2').to(device).eval()
    
    embeddings = []
    
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="    Extracting"):
            batch = texts[i:i+batch_size]
            
            # Tokenize
            inputs = tokenizer(batch, return_tensors='pt', padding=True, truncation=True, max_length=256)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            # Forward pass
            outputs = model(**inputs, output_hidden_states=True)
            
            # Extract [CLS] from Layer 12 (encoder output = last_hidden_state)
            cls_tokens = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.extend(cls_tokens)
            
            del outputs, inputs
            torch.cuda.empty_cache()
    
    # Clean up
    model.to('cpu')
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    
    return np.array(embeddings)

print(f"\nExtracting PhoBERT v2 Pretrained (Layer 12) from loose text...")
print(f"  Train texts: {len(texts_train_loose)}")
print(f"  Val texts: {len(texts_val_loose)}")
print(f"  Test texts: {len(texts_test_loose)}")
X_train_phobert = extract_phobert_embeddings(texts_train_loose, batch_size=16)
X_val_phobert = extract_phobert_embeddings(texts_val_loose, batch_size=16)
X_test_phobert = extract_phobert_embeddings(texts_test_loose, batch_size=16)

print(f"\n✅ PhoBERT embeddings extracted:")
print(f"   Train: {X_train_phobert.shape}")
print(f"   Val: {X_val_phobert.shape}")
print(f"   Test: {X_test_phobert.shape}")


EXTRACTING PHOBERT EMBEDDINGS (Layer 12 - Pretrained)

Extracting PhoBERT v2 Pretrained (Layer 12) from loose text...
  Train texts: 3788
  Val texts: 474
  Test texts: 474
    Loading model...


Some weights of RobertaModel were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
    Extracting: 100%|██████████| 237/237 [00:53<00:00,  4.40it/s]


    Loading model...


Some weights of RobertaModel were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
    Extracting: 100%|██████████| 30/30 [00:06<00:00,  4.29it/s]


    Loading model...


Some weights of RobertaModel were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
    Extracting: 100%|██████████| 30/30 [00:06<00:00,  4.32it/s]



✅ PhoBERT embeddings extracted:
   Train: (3788, 768)
   Val: (474, 768)
   Test: (474, 768)


## 5. Extract ALL Hand-Crafted Features

In [6]:
print("\n" + "="*80)
print("EXTRACTING ALL HAND-CRAFTED FEATURES (NUMERIC ONLY)")
print("="*80)

# Extract NUMERIC hand-crafted features only (exclude text columns, id, label, and non-numeric)
exclude_cols = {'id', 'label', 'text_strict', 'text_loose', 'post_message'}
numeric_cols = df_train.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [col for col in numeric_cols 
                if col not in exclude_cols and col.strip() != '' and not col.startswith('Unnamed')]

print(f"\nHand-crafted NUMERIC features found: {len(feature_cols)}")
if feature_cols:
    print(f"\nFeature list:\"")
    for i, feat in enumerate(feature_cols, 1):
        dtype = df_train[feat].dtype
        print(f"  {i:2d}. {feat:30s} [{dtype}]")
    
    # Extract features and fill NaN with 0
    X_train_features = df_train[feature_cols].fillna(0).values
    X_val_features = df_val[feature_cols].fillna(0).values
    X_test_features = df_test[feature_cols].fillna(0).values
    
    print(f"\n✅ Hand-crafted features extracted:")
    print(f"   Train: {X_train_features.shape}")
    print(f"   Val: {X_val_features.shape}")
    print(f"   Test: {X_test_features.shape}")
else:
    print(f"   ℹ️  No numeric hand-crafted features found!")
    print(f"   (Excluding: {exclude_cols})")
    X_train_features = None
    X_val_features = None
    X_test_features = None


EXTRACTING ALL HAND-CRAFTED FEATURES (NUMERIC ONLY)

Hand-crafted NUMERIC features found: 23

Feature list:"
   1. num_char                       [int64]
   2. num_emoji                      [int64]
   3. num_url                        [int64]
   4. num_hashtag                    [int64]
   5. num_like                       [int64]
   6. num_cmt                        [int64]
   7. num_share                      [int64]
   8. feat_word_count                [int64]
   9. feat_exclamation               [int64]
  10. feat_question                  [int64]
  11. feat_uppercase_words           [int64]
  12. feat_uppercase_ratio           [float64]
  13. feat_punctuation_ratio         [float64]
  14. feat_engagement_total          [int64]
  15. feat_like_ratio                [float64]
  16. feat_comment_ratio             [float64]
  17. feat_like_per_char             [float64]
  18. feat_hour_sin                  [float64]
  19. feat_hour_cos                  [float64]
  20. feat_day_sin   

## 6. Combine All Features

In [7]:
print("\n" + "="*80)
print("COMBINING ALL FEATURES")
print("="*80)

# List all components
print(f"\nFeature dimensions:")
print(f"  TF-IDF (strict):        900")
print(f"  PhoBERT (loose):        768")
print(f"  Hand-crafted features:  23")


list_train = [X_train_tfidf, X_train_phobert, X_train_features]
list_val = [X_val_tfidf, X_val_phobert, X_val_features]
list_test = [X_test_tfidf, X_test_phobert, X_test_features]

total_dims = 900 + 768 + 23

# Concatenate all
X_train = np.hstack(list_train)
X_val = np.hstack(list_val)
X_test = np.hstack(list_test)


print(f"\n{'─'*80}")
print(f"  TOTAL DIMENSIONS:       {total_dims}")

print(f"{'─'*80}")

print(f"\nCombined feature shapes:")
print(f"  Train: {X_train.shape}")
print(f"  Val:   {X_val.shape}")
print(f"  Test:  {X_test.shape}")


COMBINING ALL FEATURES

Feature dimensions:
  TF-IDF (strict):        900
  PhoBERT (loose):        768
  Hand-crafted features:  23

────────────────────────────────────────────────────────────────────────────────
  TOTAL DIMENSIONS:       1691
────────────────────────────────────────────────────────────────────────────────

Combined feature shapes:
  Train: (3788, 1691)
  Val:   (474, 1691)
  Test:  (474, 1691)


## 7. Preprocessing & Scaling

In [8]:
print("\n" + "="*80)
print("DATA PREPROCESSING")
print("="*80)

# Scale features using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"\n✅ Data scaled using StandardScaler")
print(f"   Train - Mean: {X_train_scaled.mean():.6f}, Std: {X_train_scaled.std():.6f}")
print(f"   Val   - Mean: {X_val_scaled.mean():.6f}, Std: {X_val_scaled.std():.6f}")
print(f"   Test  - Mean: {X_test_scaled.mean():.6f}, Std: {X_test_scaled.std():.6f}")


DATA PREPROCESSING

✅ Data scaled using StandardScaler
   Train - Mean: -0.000000, Std: 1.000000
   Val   - Mean: -0.000678, Std: 0.815596
   Test  - Mean: 0.000340, Std: 0.813320


## 8. Model 1: Ensemble LightGBM + CatBoost + XGBoost + MLPClassifier 

In [9]:
from sklearn.linear_model import LogisticRegression

model1a = LogisticRegression(
    penalty='l2',
    C=1.0,
    max_iter=2000,
    solver='lbfgs',
    class_weight='balanced',
    random_state=42,
    verbose=0
)
model1a.fit(X_train_scaled, y_train)
print(f"Sub-model 1a trained")

y_train_1a = model1a.predict(X_train_scaled)
y_train_1a_proba = model1a.predict_proba(X_train_scaled)[:, 1]

y_val_1a = model1a.predict(X_val_scaled)
y_val_1a_proba = model1a.predict_proba(X_val_scaled)[:, 1]

y_test_1a = model1a.predict(X_test_scaled)
y_test_1a_proba = model1a.predict_proba(X_test_scaled)[:, 1]

m1a_accuracy = accuracy_score(y_val, y_val_1a)
m1a_precision = precision_score(y_val, y_val_1a, average='weighted')
m1a_recall = recall_score(y_val, y_val_1a, average='weighted')
m1a_f1 = f1_score(y_val, y_val_1a, average='weighted')
m1a_auc = roc_auc_score(y_val, y_val_1a_proba)

print()
print(f"Accuracy:  {m1a_accuracy:.4f}")
print(f"Precision: {m1a_precision:.4f}")
print(f"Recall:    {m1a_recall:.4f}")
print(f"F1 Score:  {m1a_f1:.4f}")
print(f"AUC-ROC:   {m1a_auc:.4f}")

Sub-model 1a trained

Accuracy:  0.9093
Precision: 0.9088
Recall:    0.9093
F1 Score:  0.9091
AUC-ROC:   0.9432


In [10]:
from sklearn.ensemble import RandomForestClassifier

model1b = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=0
)
model1b.fit(X_train_scaled, y_train)
print(f"Sub-model 1b trained")

y_train_1b = model1b.predict(X_train_scaled)
y_train_1b_proba = model1b.predict_proba(X_train_scaled)[:, 1]

y_val_1b = model1b.predict(X_val_scaled)
y_val_1b_proba = model1b.predict_proba(X_val_scaled)[:, 1]

y_test_1b = model1b.predict(X_test_scaled)
y_test_1b_proba = model1b.predict_proba(X_test_scaled)[:, 1]

m1b_accuracy = accuracy_score(y_val, y_val_1b)
m1b_precision = precision_score(y_val, y_val_1b, average='weighted')
m1b_recall = recall_score(y_val, y_val_1b, average='weighted')
m1b_f1 = f1_score(y_val, y_val_1b, average='weighted')
m1b_auc = roc_auc_score(y_val, y_val_1b_proba)

print()
print(f"Accuracy:  {m1b_accuracy:.4f}")
print(f"Precision: {m1b_precision:.4f}")
print(f"Recall:    {m1b_recall:.4f}")
print(f"F1 Score:  {m1b_f1:.4f}")
print(f"AUC-ROC:   {m1b_auc:.4f}")

Sub-model 1b trained

Accuracy:  0.9008
Precision: 0.9031
Recall:    0.9008
F1 Score:  0.8865
AUC-ROC:   0.9302


In [11]:
from lightgbm import LGBMClassifier

model1c = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=7,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    is_unbalance=True,
    random_state=42,
    verbose=-1
)
model1c.fit(X_train_scaled, y_train)
print(f"Sub-model 1c trained")

y_train_1c = model1c.predict(X_train_scaled)
y_train_1c_proba = model1c.predict_proba(X_train_scaled)[:, 1]

y_val_1c = model1c.predict(X_val_scaled)
y_val_1c_proba = model1c.predict_proba(X_val_scaled)[:, 1]

y_test_1c = model1c.predict(X_test_scaled)
y_test_1c_proba = model1c.predict_proba(X_test_scaled)[:, 1]

m1c_accuracy = accuracy_score(y_val, y_val_1c)
m1c_precision = precision_score(y_val, y_val_1c, average='weighted')
m1c_recall = recall_score(y_val, y_val_1c, average='weighted')
m1c_f1 = f1_score(y_val, y_val_1c, average='weighted')
m1c_auc = roc_auc_score(y_val, y_val_1c_proba)

print()
print(f"Accuracy:  {m1c_accuracy:.4f}")
print(f"Precision: {m1c_precision:.4f}")
print(f"Recall:    {m1c_recall:.4f}")
print(f"F1 Score:  {m1c_f1:.4f}")
print(f"AUC-ROC:   {m1c_auc:.4f}")

Sub-model 1c trained

Accuracy:  0.9177
Precision: 0.9142
Recall:    0.9177
F1 Score:  0.9148
AUC-ROC:   0.9381


In [12]:
from catboost import CatBoostClassifier

model1d = CatBoostClassifier(
    iterations=200,
    learning_rate=0.05,
    depth=7,
    auto_class_weights='Balanced',
    random_state=42,
    verbose=0
)
model1d.fit(X_train_scaled, y_train)
print(f"Sub-model 1d trained")

y_train_1d = model1d.predict(X_train_scaled)
y_train_1d_proba = model1d.predict_proba(X_train_scaled)[:, 1]

y_val_1d = model1d.predict(X_val_scaled)
y_val_1d_proba = model1d.predict_proba(X_val_scaled)[:, 1]

y_test_1d = model1d.predict(X_test_scaled)
y_test_1d_proba = model1d.predict_proba(X_test_scaled)[:, 1]

m1d_accuracy = accuracy_score(y_val, y_val_1d)
m1d_precision = precision_score(y_val, y_val_1d, average='weighted')
m1d_recall = recall_score(y_val, y_val_1d, average='weighted')
m1d_f1 = f1_score(y_val, y_val_1d, average='weighted')
m1d_auc = roc_auc_score(y_val, y_val_1d_proba)

print()
print(f"Accuracy:  {m1d_accuracy:.4f}")
print(f"Precision: {m1d_precision:.4f}")
print(f"Recall:    {m1d_recall:.4f}")
print(f"F1 Score:  {m1d_f1:.4f}")
print(f"AUC-ROC:   {m1d_auc:.4f}")

Sub-model 1d trained

Accuracy:  0.9135
Precision: 0.9180
Recall:    0.9135
F1 Score:  0.9153
AUC-ROC:   0.9402


In [13]:
from xgboost import XGBClassifier

model1e = XGBClassifier(
    n_estimators=200,         
    learning_rate=0.05,      
    max_depth=7,                
    subsample=0.8,            
    colsample_bytree=0.8,      
    reg_alpha=0.1,             
    reg_lambda=0.1,            
    scale_pos_weight=1,         
    random_state=42,
    use_label_encoder=False,    
    eval_metric='logloss'      
)
model1e.fit(X_train_scaled, y_train)
print(f"Sub-model 1e trained")

y_train_1e = model1e.predict(X_train_scaled)
y_train_1e_proba = model1e.predict_proba(X_train_scaled)[:, 1]

y_val_1e = model1e.predict(X_val_scaled)
y_val_1e_proba = model1e.predict_proba(X_val_scaled)[:, 1]

y_test_1e = model1e.predict(X_test_scaled) 
y_test_1e_proba = model1e.predict_proba(X_test_scaled)[:, 1]

m1e_accuracy = accuracy_score(y_val, y_val_1e)
m1e_precision = precision_score(y_val, y_val_1e, average='weighted')
m1e_recall = recall_score(y_val, y_val_1e, average='weighted')
m1e_f1 = f1_score(y_val, y_val_1e, average='weighted')
m1e_auc = roc_auc_score(y_val, y_val_1e_proba)

print()
print(f"Accuracy:  {m1e_accuracy:.4f}")
print(f"Precision: {m1e_precision:.4f}")
print(f"Recall:    {m1e_recall:.4f}")
print(f"F1 Score:  {m1e_f1:.4f}")
print(f"AUC-ROC:   {m1e_auc:.4f}")

Sub-model 1e trained

Accuracy:  0.9114
Precision: 0.9104
Recall:    0.9114
F1 Score:  0.9022
AUC-ROC:   0.9409


In [14]:
from sklearn.neural_network import MLPClassifier

model1f = MLPClassifier(
hidden_layer_sizes=(128, 64),
activation='relu',
solver='adam',
learning_rate='constant',
learning_rate_init=0.001,
max_iter=300,
early_stopping=True,
validation_fraction=0.1,
n_iter_no_change=10,
batch_size=32,
random_state=42,
verbose=False,
)



model1f.fit(X_train_scaled, y_train)
print(f"Sub-model 1f trained")

y_train_1f = model1f.predict(X_train_scaled)
y_train_1f_proba = model1f.predict_proba(X_train_scaled)[:, 1]

y_val_1f = model1f.predict(X_val_scaled)
y_val_1f_proba = model1f.predict_proba(X_val_scaled)[:, 1]

y_test_1f = model1f.predict(X_test_scaled)
y_test_1f_proba = model1f.predict_proba(X_test_scaled)[:, 1]

m1f_accuracy = accuracy_score(y_val, y_val_1f)
m1f_precision = precision_score(y_val, y_val_1f, average='weighted')
m1f_recall = recall_score(y_val, y_val_1f, average='weighted')
m1f_f1 = f1_score(y_val, y_val_1f, average='weighted')
m1f_auc = roc_auc_score(y_val, y_val_1f_proba)

print()
print(f"Accuracy:  {m1f_accuracy:.4f}")
print(f"Precision: {m1f_precision:.4f}")
print(f"Recall:    {m1f_recall:.4f}")
print(f"F1 Score:  {m1f_f1:.4f}")
print(f"AUC-ROC:   {m1f_auc:.4f}")

Sub-model 1f trained

Accuracy:  0.9241
Precision: 0.9241
Recall:    0.9241
F1 Score:  0.9241
AUC-ROC:   0.9567


In [15]:
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn   
import torch.optim as optim
import time
import copy

class LSTMNet(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2, dropout=0.5):
        super(LSTMNet, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 2)
        )
    
    def forward(self, x):
        # x shape: (batch, seq_len=1, input_dim)
        lstm_out, (h_n, c_n) = self.lstm(x)  # h_n: (num_layers, batch, hidden_dim)
        # Use last hidden state from top layer
        last_hidden = h_n[-1]  # (batch, hidden_dim)
        out = self.fc(last_hidden)  # (batch, 2)
        return out

print("✅ LSTM architecture defined")

# Create tensors and DataLoaders
train_dataset_m1g = TensorDataset(torch.from_numpy(X_train_scaled).float().unsqueeze(1),
                                   torch.from_numpy(y_train).long())
val_dataset_m1g = TensorDataset(torch.from_numpy(X_val_scaled).float().unsqueeze(1),
                                 torch.from_numpy(y_val).long())

train_loader_m1g = DataLoader(train_dataset_m1g, batch_size=42, shuffle=True)
val_loader_m1g = DataLoader(val_dataset_m1g, batch_size=42, shuffle=False)

# Initialize and train LSTM Model 1g
model1g = LSTMNet(input_dim=X_train_scaled.shape[1]).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model1g.parameters(), lr=0.001)
start_time = time.time()


best_auc = -float("inf")
best_epoch = -1
best_model_state = None

for epoch in range(50):
    # Training
    model1g.train()
    train_loss = 0.0
    for X_batch, y_batch in train_loader_m1g:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        pred = model1g(X_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Validation
    model1g.eval()
    val_loss = 0.0
    all_preds, all_preds_proba, all_labels = [], [], []

    with torch.no_grad():
        for X_batch, y_batch in val_loader_m1g:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            pred = model1g(X_batch)
            loss = criterion(pred, y_batch)
            val_loss += loss.item()
            all_preds.extend(pred.argmax(1).cpu().numpy())
            all_preds_proba.extend(torch.softmax(pred, dim=1)[:, 1].cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    val_acc = accuracy_score(all_labels, all_preds)
    val_auc = roc_auc_score(all_labels, all_preds_proba)

    # Track best model by AUC (deep copy)
    if val_auc > best_auc:
        best_auc = val_auc
        best_epoch = epoch + 1
        best_model_state = copy.deepcopy(model1g.state_dict())

    if (epoch + 1) % 5 == 0:
        print(f"  Epoch {epoch+1:2d}: Train Loss={train_loss/len(train_loader_m1g):.4f}, "
              f"Val Loss={val_loss/len(val_loader_m1g):.4f}, Val Acc={val_acc:.4f}, Val AUC={val_auc:.4f}")

# Restore best model
model1g.load_state_dict(best_model_state)
print(f"✅ Loaded best model from Epoch {best_epoch} (AUC={best_auc:.4f})")

train_time_m1g = time.time() - start_time
print(f"✅ Model 1g training completed in {train_time_m1g:.2f}s\n")

# ============================================================================
# EVALUATE MODEL 1G - ALL METRICS
# ============================================================================
print("\n" + "="*80)
print("MODEL 1G: FINAL EVALUATION")
print("="*80)

model1g.eval()
with torch.no_grad():
    X_train_m1g_tensor = torch.from_numpy(X_train_scaled).float().unsqueeze(1).to(device)
    outputs_train = model1g(X_train_m1g_tensor)
    y_train_1g = outputs_train.argmax(dim=1).cpu().numpy()
    y_train_1g_proba = torch.softmax(outputs_train, dim=1)[:, 1].cpu().numpy()

    X_val_m1g_tensor = torch.from_numpy(X_val_scaled).float().unsqueeze(1).to(device)
    outputs_val = model1g(X_val_m1g_tensor)
    y_val_1g = outputs_val.argmax(dim=1).cpu().numpy()
    y_val_1g_proba = torch.softmax(outputs_val, dim=1)[:, 1].cpu().numpy()

    X_test_m1g_tensor = torch.from_numpy(X_test_scaled).float().unsqueeze(1).to(device)
    outputs_test = model1g(X_test_m1g_tensor)
    y_test_1g = outputs_test.argmax(dim=1).cpu().numpy()
    y_test_1g_proba = torch.softmax(outputs_test, dim=1)[:, 1].cpu().numpy()

m1g_accuracy = accuracy_score(y_val, y_val_1g)
m1g_precision = precision_score(y_val, y_val_1g, average='weighted')
m1g_recall = recall_score(y_val, y_val_1g, average='weighted')
m1g_f1 = f1_score(y_val, y_val_1g, average='weighted')
m1g_auc = roc_auc_score(y_val, y_val_1g_proba)


print()
print(f"Accuracy:  {m1g_accuracy:.4f}")
print(f"Precision: {m1g_precision:.4f}")
print(f"Recall:    {m1g_recall:.4f}")
print(f"F1 Score:  {m1g_f1:.4f}")
print(f"AUC-ROC:   {m1g_auc:.4f}")


✅ LSTM architecture defined
  Epoch  5: Train Loss=0.0155, Val Loss=0.4011, Val Acc=0.9177, Val AUC=0.9513
  Epoch 10: Train Loss=0.0158, Val Loss=0.4655, Val Acc=0.9262, Val AUC=0.9434
  Epoch 15: Train Loss=0.0143, Val Loss=0.3825, Val Acc=0.9304, Val AUC=0.9564
  Epoch 20: Train Loss=0.0078, Val Loss=0.4984, Val Acc=0.9241, Val AUC=0.9485
  Epoch 25: Train Loss=0.0058, Val Loss=0.6442, Val Acc=0.9177, Val AUC=0.9419
  Epoch 30: Train Loss=0.0101, Val Loss=0.5662, Val Acc=0.9114, Val AUC=0.9422
  Epoch 35: Train Loss=0.0048, Val Loss=0.6278, Val Acc=0.9177, Val AUC=0.9437
  Epoch 40: Train Loss=0.0060, Val Loss=0.5888, Val Acc=0.9198, Val AUC=0.9504
  Epoch 45: Train Loss=0.0063, Val Loss=0.6296, Val Acc=0.9072, Val AUC=0.9533
  Epoch 50: Train Loss=0.0077, Val Loss=0.5986, Val Acc=0.9177, Val AUC=0.9477
✅ Loaded best model from Epoch 15 (AUC=0.9564)
✅ Model 1g training completed in 14.40s


MODEL 1G: FINAL EVALUATION

Accuracy:  0.9304
Precision: 0.9280
Recall:    0.9304
F1 Score:  

In [16]:
# Ensemble: Average probabilities from 1a, 1b, 1c, 1d, 1e, 1f, and 1g (LSTM)
print("\n" + "="*80)
print("ENSEMBLE MODEL: AVERAGE PROBABILITIES")
print("="*80)
y_val_proba_ensemble = (y_val_1a_proba + y_val_1b_proba + y_val_1c_proba + y_val_1d_proba + y_val_1e_proba + y_val_1f_proba + y_val_1g_proba) / 7
y_val_pred_ensemble = (y_val_proba_ensemble > 0.5).astype(int)

ensemble_accuracy = accuracy_score(y_val, y_val_pred_ensemble)
ensemble_precision = precision_score(y_val, y_val_pred_ensemble, average='weighted')
ensemble_recall = recall_score(y_val, y_val_pred_ensemble, average='weighted')
ensemble_f1 = f1_score(y_val, y_val_pred_ensemble, average='weighted')
ensemble_auc = roc_auc_score(y_val, y_val_proba_ensemble)

print(f"\n📊 5-Model Ensemble (LightGBM + CatBoost + XGBoost + MLP + LSTM):")
print(f"   Accuracy:  {ensemble_accuracy:.4f}")
print(f"   Precision: {ensemble_precision:.4f}")
print(f"   Recall:    {ensemble_recall:.4f}")
print(f"   F1 Score:  {ensemble_f1:.4f}")
print(f"   AUC-ROC:   {ensemble_auc:.4f}")

print("\n" + "="*80)



ENSEMBLE MODEL: AVERAGE PROBABILITIES

📊 5-Model Ensemble (LightGBM + CatBoost + XGBoost + MLP + LSTM):
   Accuracy:  0.9304
   Precision: 0.9279
   Recall:    0.9304
   F1 Score:  0.9279
   AUC-ROC:   0.9586



In [17]:
y_test_1a_proba = model1a.predict_proba(X_test_scaled)[:, 1]

y_test_1b_proba = model1b.predict_proba(X_test_scaled)[:, 1]

y_test_1c_proba = model1c.predict_proba(X_test_scaled)[:, 1]

y_test_1d_proba = model1d.predict_proba(X_test_scaled)[:, 1]

y_test_1e_proba = model1e.predict_proba(X_test_scaled)[:, 1]

y_test_1f_proba = model1f.predict_proba(X_test_scaled)[:, 1]


# Ensemble: Average probabilities from 1a, 1b, 1c, 1d, 1e, and 1f
y_test_proba_m1 = (y_test_1a_proba + y_test_1b_proba + y_test_1c_proba + y_test_1d_proba + y_test_1e_proba + y_test_1f_proba + y_test_1g_proba) / 7
y_test_pred_m1 = (y_test_proba_m1 > 0.5).astype(int)

m1_accuracy = accuracy_score(y_test, y_test_pred_m1)
m1_precision = precision_score(y_test, y_test_pred_m1, average='weighted')
m1_recall = recall_score(y_test, y_test_pred_m1, average='weighted')
m1_f1 = f1_score(y_test, y_test_pred_m1, average='weighted')
m1_auc = roc_auc_score(y_test, y_test_proba_m1)

print(f"Accuracy:  {m1_accuracy:.4f}")
print(f"Precision: {m1_precision:.4f}")
print(f"Recall:    {m1_recall:.4f}")
print(f"F1 Score:  {m1_f1:.4f}")
print(f"AUC-ROC:   {m1_auc:.4f}")

Accuracy:  0.9008
Precision: 0.8949
Recall:    0.9008
F1 Score:  0.8940
AUC-ROC:   0.9453


In [18]:
from sklearn.linear_model import LogisticRegression

print("\n" + "="*80)
print("STACKING META MODEL: LOGISTIC REGRESSION")
print("="*80)

# Build meta features from base-model probabilities on validation
X_meta_train = np.column_stack([
    y_train_1a_proba, y_train_1b_proba, y_train_1c_proba,
    y_train_1d_proba, y_train_1e_proba, y_train_1f_proba, y_train_1g_proba
])

X_meta_val = np.column_stack([
    y_val_1a_proba, y_val_1b_proba, y_val_1c_proba,
    y_val_1d_proba, y_val_1e_proba, y_val_1f_proba, y_val_1g_proba
])

X_meta_test = np.column_stack([
    y_test_1a_proba, y_test_1b_proba, y_test_1c_proba,
    y_test_1d_proba, y_test_1e_proba, y_test_1f_proba, y_test_1g_proba
])

# Train meta model
meta_model = LogisticRegression(
    penalty='l2',
    C=1.0,
    solver='lbfgs',
    class_weight='balanced',
    max_iter=2000,
    random_state=42
 )
meta_model.fit(X_meta_train, y_train)
print("Meta-model trained on validation predictions")

# Predict on validation meta features
y_val_meta_pred = meta_model.predict(X_meta_val)
y_val_meta_score = meta_model.predict_proba(X_meta_val)[:, 1]

val_stack_accuracy = accuracy_score(y_val, y_val_meta_pred)
val_stack_precision = precision_score(y_val, y_val_meta_pred, average='weighted')
val_stack_recall = recall_score(y_val, y_val_meta_pred, average='weighted')
val_stack_f1 = f1_score(y_val, y_val_meta_pred, average='weighted')
val_stack_auc = roc_auc_score(y_val, y_val_meta_score)

print()
print(f"Stacking Accuracy:  {val_stack_accuracy:.4f}")
print(f"Stacking Precision: {val_stack_precision:.4f}")
print(f"Stacking Recall:    {val_stack_recall:.4f}")
print(f"Stacking F1 Score:  {val_stack_f1:.4f}")
print(f"Stacking AUC-ROC:   {val_stack_auc:.4f}")

# Predict on test meta features
y_test_meta_pred = meta_model.predict(X_meta_test)
y_test_meta_score = meta_model.predict_proba(X_meta_test)[:, 1]

test_stack_accuracy = accuracy_score(y_test, y_test_meta_pred)
test_stack_precision = precision_score(y_test, y_test_meta_pred, average='weighted')
test_stack_recall = recall_score(y_test, y_test_meta_pred, average='weighted')
test_stack_f1 = f1_score(y_test, y_test_meta_pred, average='weighted')
test_stack_auc = roc_auc_score(y_test, y_test_meta_score)

print()
print(f"Test Stacking Accuracy:  {test_stack_accuracy:.4f}")
print(f"Test Stacking Precision: {test_stack_precision:.4f}")
print(f"Test Stacking Recall:    {test_stack_recall:.4f}")
print(f"Test Stacking F1 Score:  {test_stack_f1:.4f}")
print(f"Test Stacking AUC-ROC:   {test_stack_auc:.4f}")


STACKING META MODEL: LOGISTIC REGRESSION
Meta-model trained on validation predictions

Stacking Accuracy:  0.9325
Stacking Precision: 0.9302
Stacking Recall:    0.9325
Stacking F1 Score:  0.9299
Stacking AUC-ROC:   0.9590

Test Stacking Accuracy:  0.9030
Test Stacking Precision: 0.8975
Test Stacking Recall:    0.9030
Test Stacking F1 Score:  0.8980
Test Stacking AUC-ROC:   0.9451
